In [1]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%matplotlib inline

import os
import sys
import torch
import matplotlib
import matplotlib.pyplot as plt

if not '../src' in sys.path:
    sys.path.append('../src')

from dataset import MuonDataset
from mappers import *
from surrogates import *
from tools import *
from propagation import *
from scheduler import *
from callbacks import *
from constants import *

In [3]:
def selection(ri,rf,di,df):
    distance_mask =  (ri-rf).norm(dim=-1) > 0.05
    angle = torch.rad2deg(
        torch.acos(
            (di*df).sum(dim=-1)
        )
    )
    angle_mask = torch.logical_and(
        ~torch.isnan(angle),
        angle < 80.,
    )
    mask = torch.logical_and(distance_mask,angle_mask)
    return mask
dataset = MuonDataset.from_data_file(
    data_file = '/cephfs/dice/users/lw23382/MuonTransmissionData/data_obj1.npy',
    momentum_file = '/cephfs/dice/users/lw23382/MuonTransmissionData/momenta_obj1.npy',
    selection = selection,
)

Masking 31765 out of 3844288 events [0.826%]


In [4]:
surrogate = LossSurrogate(
    flow = NormalizingFlow(
        features = dataset.observable_dim,
        context = dataset.condition_dim,
        hidden_features = [256,256],
        transforms = 5,
        bins = 16,
    ),
)
# # surrogate = torch.load('surrogate.pt',weights_only=False,map_location='cpu')
# # print (surrogate)

In [5]:
scheduler = EnergyDistanceScheduler(
    dataset = dataset,
    mapper = VoxelisedMapper(
        xlow = dataset.low,
        xhigh = dataset.high,
        ylow = LAMBDA_MIN,
        yhigh = LAMBDA_MAX,
        resolution = [8,16,8],
        initial_val = 1./X0s['Air'],
        lr = 1e-3,
        alpha_tv = 1e-3,
    ),
    propagator = RossiPropagator(
        step_size = 0.01, # m
    ),
    data_surrogate = surrogate, #torch.load('data_surrogate.pt',weights_only=False,map_location='cpu'),
    n_samples = 10,
    device = 'cuda:5',
    callbacks = [
        LossPlotCallback(frequency=5,log=True),
        MapperPlotCallback(
            low = dataset.low,
            high = dataset.high,
            frequency = 5,
        ),
        ObservablesPlotCallback(frequency=5),
    ],
    gamma = 1.,
    p = 2.,
    batch_size = 500,
)
print (scheduler.mapper)

RuntimeError: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.

In [ ]:
scheduler.train_data_surrogate()

In [ ]:
from constants import *
print (X0s)
for i,(name,X0) in enumerate(X0s.items()):
    lam = 1/X0
    print (name,i,lam)
    plt.scatter(
        i,
        lam,
        label = name,
    )
plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
# def sampling_function(batch_size):
#     low = dataset.low - 10
#     high  = dataset.high + 10
#     x = low + torch.rand(batch_size,3) * (high-low)
#     lam = 1/X0s['Fe']
#     return x,lam * torch.ones((batch_size,1))

# scheduler.mapper.set_lr(1e-2)
# scheduler.pre_train_mapper(
#     sampling_function = sampling_function,
#     loss_cutoff = 1e-15,
#     batch_size = 256,
#     max_epochs = 20000,
# )
# scheduler.mapper.set_lr(1e-3)
# scheduler.pre_train_mapper(
#     sampling_function = sampling_function,
#     loss_cutoff = 1e-16,
#     batch_size = 256,
#     max_epochs = 10000,
# )
# scheduler.mapper.set_lr(1e-4)
# scheduler.pre_train_mapper(
#     sampling_function = sampling_function,
#     loss_cutoff = 1e-17,
#     batch_size = 256,
#     max_epochs = 10000,
# )
# scheduler.mapper.set_lr(1e-5)
# scheduler.pre_train_mapper(
#     sampling_function = sampling_function,
#     loss_cutoff = 1e-17,
#     batch_size = 256,
#     max_epochs = 10000,
# )

In [ ]:
fig = scheduler.mapper.plot_slice(
    direction = 'y',
    value = 3.0,
    low = [dataset.low[0],dataset.low[2]],
    high = [dataset.high[0],dataset.high[2]],
    batch_size = 5000,
    points = 50,
    show_materials = True,
)
plt.show()

In [ ]:
# # scheduler.n_samples=1
# sim_datasets,idx = scheduler.propagate(1000,no_grad=True)

# fig = scheduler.callbacks[2].on_observables_end(None,sim_datasets[0].observables,dataset.observables,dataset.observable_names)
# plt.show()

# sim_datasets[0].plot_3d(
#     N = 20,
#     arrow_scale=0.5,
#     show_direction=True,
#     show_path=True,                                                                                                                                                                                              
# )
# plt.show()

In [ ]:
scheduler.mapper.set_lr(1e-0)

In [ ]:
scheduler.optimize(n_iterations=1000)

In [ ]:
scheduler.callbacks[0].plot(iteration=None)

In [ ]:
scheduler.callbacks[1].plot(iteration=None,mapper=scheduler.mapper)

In [ ]:
fig = mapper.plot_slice(
    direction = 'z',
    value = 2.5,
    low = dataset.low[:2],
    high = dataset.high[:2],
    batch_size = 5000,
    points = 20,
)
plt.show()

fig = mapper.plot_slice(
    direction = 'y',self.propagate(batch_size)
    value = 3.0,
    low = [dataset.low[0],dataset.low[2]],
    high = [dataset.high[0],dataset.high[2]],
    batch_size = 5000,
    points = 20,
)
plt.show()

In [ ]:
fig = mapper.plot_slice(
    direction = 'z',
    value = 2.5,
    low = dataset.low[:2],
    high = dataset.high[:2],
    batch_size = 5000,
    points = 20,
)
plt.show()

fig = mapper.plot_slice(
    direction = 'y',
    value = 3.0,
    low = [dataset.low[0],dataset.low[2]],
    high = [dataset.high[0],dataset.high[2]],
    batch_size = 5000,
    points = 20,
)
plt.show()

In [ ]:
scheduler.run(n_iterations=100,batch_size=256)

In [ ]:
scheduler.surrogate.validate_model(batch_size=1000)

In [ ]:
observables = positions_to_observables(
    ri = ri,
    rf = rf,
    di = di,
    df = df,
    lo = dataset.low,
    hi = dataset.high,
)
observables = torch.stack(observables,dim=1)
conditions = torch.cat([ri,di],dim=1)


In [ ]:
fig = scheduler.compare_observables(observables.detach())
plt.show()